# V-KPI Admin 全系统审计：可复跑数据附录

## tl;dr

- 系统广度已经足够；当前瓶颈是事实口径、新鲜度、执行转化和结果标签。
- 该 notebook 使用本轮只读数据库快照，复核报告里的关键数字。
- 浏览器视觉证据位于同目录 PNG，代码基线为 `9bfcd4a944104e6b010a93df50d07ed7e5c44847`。

## Context & Methods

本附录把公司账号时序、Action Inbox、开放告警、Projects、回复队列、团队协作和结果标签放到同一个审计快照。

### Key Assumptions

- 数据库 `CURRENT_DATE` 使用运行实例时区，因此新鲜度与 America/New_York 可能相差 1 天。
- `executed / all actions` 只表示状态转化，不等于业务结果成功。
- 页面健康分是基于本轮 Admin 实机、数据可用性和闭环完整性的审计评分，不是用户满意度调查。

In [1]:
import json
from pathlib import Path
import pandas as pd

SNAPSHOT_PATH = Path('audit_snapshot.json')
snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding='utf-8'))
data = snapshot['datasets']
print('datasets:', ', '.join(sorted(data)))
print('generatedAt:', snapshot['generatedAt'])

datasets: action_status, assignment_stages, daily_channel_metrics, flatness, headline, open_alerts, page_audit, project_stages, reply_status, system_scores
generatedAt: 2026-07-10T04:16:17+00:00


In [2]:
headline = pd.DataFrame(data['headline'])
print(headline.to_string(index=False))

 metric_rows  channels metric_latest_date  data_age_days narrative_latest_date  worker_stale_days  actions_total  actions_executed  action_execution_pct  open_alerts  staff_total  staff_no_last_active  staff_groups  collab_settings  gtm_outcomes  prediction_evals  outcome_evaluations  successful_evaluations  distinct_success_labels  events  event_tasks  event_evidence  event_retrospectives  dealers  shopify_orders  goaffpro_sales
         502        18         2026-06-14             26            2026-06-29                6.7            291                13                   4.5           74           20                    20             0                0             0                 0                   98                      98                        1       5            0               2                     2        0               0               0


In [3]:
flatness = pd.DataFrame(data['flatness'])
print(flatness.to_string(index=False))
print('Interpretation: likes/comments are unchanged in most consecutive channel snapshots, so apparent daily coverage does not imply fresh interaction data.')

 pairs  flat_likes  flat_likes_pct  flat_comments  flat_comments_pct
   484         463            95.7            457               94.4
Interpretation: likes/comments are unchanged in most consecutive channel snapshots, so apparent daily coverage does not imply fresh interaction data.


In [4]:
actions = pd.DataFrame(data['action_status'])
print(actions.to_string(index=False))
executed = float(actions.loc[actions.status.eq('executed'), 'share_pct'].iloc[0])
print(f'Executed share: {executed:.1f}%')

   status  count  share_pct
suggested    196       67.4
dismissed     75       25.8
 executed     13        4.5
 approved      6        2.1
  snoozed      1        0.3
Executed share: 4.5%


In [5]:
daily = pd.DataFrame(data['daily_channel_metrics'])
daily['date'] = pd.to_datetime(daily['date'])
print(daily.tail(12)[['date','channels','followers','views_delta_24h','total_likes','total_comments']].to_string(index=False))

      date  channels  followers  views_delta_24h  total_likes  total_comments
2026-06-03        18    1195786           735010     14462617          263791
2026-06-04        18    1196414           700676     14462617          263791
2026-06-05        18    1196960           744071     14462617          263791
2026-06-06        18    1197517           503148     14462617          263791
2026-06-07        18    1198381           373655     14462617          263791
2026-06-08        18    1199260           404045     14462617          263791
2026-06-09        18    1199802          1277296     14462617          263791
2026-06-10        18    1201068          1591963     14462617          263791
2026-06-11        18    1202169          1927485     14462617          263791
2026-06-12        18    1203332          1484549     14462617          263791
2026-06-13        18    1204327          1516781     14462617          263791
2026-06-14        18    1205115          1401838     14462617   

In [6]:
pages = pd.DataFrame(data['page_audit']).sort_values(['priority','health'])
print(pages[['area','health','priority','finding']].to_string(index=False))

              area  health priority                          finding
             创意资产库      20       P0                实机持续停留在检索中，核心内容空白
            Events      24       P0         测试数据、NaN、9642 天活动；任务表为 0
              回复队列      29       P0 132 条中 130 pending，语言/意图识别存在明显误判
全局搜索 / Intelligent      30       P0       复杂经营问题被误判成固定 ROI 排名并返回 0 行
              团队管理      32       P0         20 人但无活跃/登录时间、0 组、0 协作设置
            Report      38       P0       Dashboard 与报告人数、曝光、互动率口径冲突
           提醒 / 通知      40       P0      提醒 0、建议 12、未读 74；模型分裂且有审计噪音
              自治驾照      44       P0        治理概念强，但样本 n=2 或 0，暂不应升级自治
            MY KOL      45       P0   20 人、18 账号、717 MY KOL、1 个库对象并存
       GTM Command      47       P0  治理信息完整，但 North Star 几乎未推进且结果为 0
         Dashboard      52       P0       指标可见但新鲜度不足，地图占首屏，Worker 离线
          Projects      58       P0   阶段可管理，但导入态、占位 tracking 与真实证据混杂
           Dealers      18       P1                   0 数据且地图瓦片未正常显示
Shopify / GOAFFPRO      28       P

In [7]:
assignments = pd.DataFrame(data['assignment_stages'])
replies = pd.DataFrame(data['reply_status'])
print('Assignment stages:')
print(assignments.to_string(index=False))
print('\nReply queue:')
print(replies.to_string(index=False))

Assignment stages:
         stage  count  placeholder_count
   device_sent    842                  0
content_posted    695                613
     contacted    209                  0
    discovered    170                  0
       churned    160                  0
        agreed     39                  0
       replied     33                  0
     discovery     23                  0
       shipped      7                  0
     cancelled      6                  0
       arrived      2                  0
      received      2                  2
      measured      1                  1

Reply queue:
 status  count
pending    130
drafted      1
replied      1


## Results

1. 公司账号明细最新到 2026-06-14，而叙事日报标到 2026-06-29，需在 UI 中同时显示 source date 与 report date。
2. Action Inbox 只有约 4.5% 进入 executed；130/132 条回复仍在 pending。
3. 20 位成员全部缺 last_active_at / last_login_at，且 0 组、0 协作设置，不能据此衡量成员互动。
4. 98 个 outcome evaluation 全部 success，标签只有一个取值，不能支撑自治升级。

## Takeaways

第一阶段应建设统一事实层和 KOL commitment supervisor；第二阶段再做问数到可追溯报告；第三阶段才扩大自动执行与学习。